In [1]:
import os
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langsmith import Client, traceable
from langsmith.evaluation import evaluate, LangChainStringEvaluator

In [ ]:
import os
os.environ['OPENAI_API_BASE'] = "https://openai.vocareum.com/v1"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "LangSmith_Evaluations_Demo_19Jul_v1"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

In [3]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b

In [10]:
@traceable
def target_agent_runner(inputs: dict) -> dict:
    """Wraps the agent execution so LangSmith can map inputs to outputs."""
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    tools = [add_numbers, multiply_numbers]
    SYSTEM_PROMPT = """ You are a helpful mathematics wizard who uses tools to do maths calculations. Only use the tools and no
    other information. Respond politely and if you do not know, say you do not know. Only give the output of the calculation.
    """
    prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT),
                                               ("human",  "{query}"),
                                               MessagesPlaceholder(variable_name="agent_scratchpad")])
    query = inputs["query"]
    agent = create_tool_calling_agent(llm=model, tools=tools, prompt=prompt)
    executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
    result = executor.invoke({"query": f"{query}"})
    return result.get('output','')

In [11]:
client = Client()
dataset_name = "Demo Mathematics Evaluation Dataset V2"
eval_data = [{"input": {"query": "What is 10 plus 20?"}, "reference": {"output": "30"}},
             {"input": {"query": "Multiply 5 by 6, then add 2."}, "reference": {"output": "32"}},
             {"input": {"query": "What is 15 plus 15?"}, "reference": {"output": "30"}},
            ]

In [12]:
if not client.has_dataset(dataset_name=dataset_name):
    print(f"Creating dataset: {dataset_name}...")
    dataset = client.create_dataset(dataset_name=dataset_name, description="Math agent test questions.")
    for item in eval_data:
        client.create_example(inputs=item["input"],
                              outputs=item["reference"],
                              dataset_id=dataset.id
                             )
else:
    print(f"Dataset '{dataset_name}' already exists. Using existing.")

Dataset 'Demo Mathematics Evaluation Dataset V2' already exists. Using existing.


In [13]:
def contains_reference_number(run, example) -> dict:
    # 1. Safely extract values and enforce string types
    # If run.outputs is None or 'output' isn't a string, fallback to an empty string
    expected = str(example.outputs.get("output", "")) if example.outputs else ""
    
    if run.outputs and "output" in run.outputs and run.outputs["output"] is not None:
        actual = str(run.outputs["output"])
    else:
        actual = ""

    # 2. Add defensive check: if actual is empty, it's a structural failure (Score: 0)
    if not actual or not expected:
        return {"key": "correct_number", "score": 0, "comment": "Missing run or reference output"}

    # 3. Perform string match safely
    score = 1 if expected in actual else 0
    return {"key": "correct_number", "score": score}

In [14]:
qa_evaluator = LangChainStringEvaluator("qa", config={"llm": ChatOpenAI(model="gpt-4o-mini", temperature=0)})

In [15]:
print("Starting LangSmith evaluation run...")
experiment_results = evaluate(target_agent_runner,               # The traceable function executing your agent
                              data=dataset_name,                 # The LangSmith dataset name
                              evaluators=[contains_reference_number, qa_evaluator], # Combined list of evaluators
                              experiment_prefix="demo-agent-math-test"  # Descriptive prefix inside dashboard
                             )
print("Evaluation completed successfully!")

Starting LangSmith evaluation run...
View the evaluation results for experiment: 'demo-agent-math-test-287526cb' at:
https://smith.langchain.com/o/ec6b70d9-0f53-4864-a145-afdeecd76dc1/datasets/b830ebf8-bbed-4223-9320-6a73d7b7b666/compare?selectedSessions=fbc11472-7154-4f13-b6fc-f54e47776ee9




0it [00:00, ?it/s]Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `add_numbers` with `{'a': 15, 'b': 15}`


30
Invoking: `add_numbers` with `{'a': 10, 'b': 20}`


30
Invoking: `multiply_numbers` with `{'a': 5, 'b': 6}`


30
Invoking: `add_numbers` with `{'a': 30, 'b': 2}`


3230

> Finished chain.
30

> Finished chain.
The result of multiplying 5 by 6 is 30, and adding 2 gives 32.

> Finished chain.


3it [00:06,  2.26s/it]

Evaluation completed successfully!
